In [ ]:
from dotenv import load_dotenv
import os
import json

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from transformers import pipeline


In [12]:

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2405.64it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo

In [13]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4591.64it/s]


In [19]:
vector_db = FAISS.load_local(
    "vector_db",
    embedding_model,
    allow_dangerous_deserialization=True
)
print(vector_db)

In [27]:
def retrieve(query, top_k=3):

    docs = vector_db.similarity_search_with_score(
        query,
        k=top_k
    )

    return docs

In [28]:
def answer_query(query):

    retrieved_docs = retrieve(query)

    context = ""

    retrieved_chunks = []

    for doc, score in retrieved_docs:

        context += doc.page_content + "\n\n"

        retrieved_chunks.append(
            {
                "page": doc.metadata.get("page", "Unknown"),
                "score": float(score)
            }
        )

    prompt = f"""
You are a question answering assistant.

Answer ONLY from the context below.

If the answer is not present, reply exactly:

I don't know from the document.

Context:
{context}

Question:
{query}

Answer:
"""

    result = generator(prompt)

    answer = result[0]["generated_text"].strip()

    return {
        "query": query,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }

In [25]:
import json

with open("queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

print(queries)

[{'query': 'What were the main causes for increased lead times mentioned in the report?'}, {'query': 'List the KPIs for Region B from the Q3 table.'}, {'query': 'Which parts have lead time more than 30 days?'}, {'query': 'What is the procedure for critical parts exceptions and the SLA for documenting root cause?'}]


In [29]:
with open("queries.json", "r", encoding="utf-8") as f:

    queries = json.load(f)

print(queries)

[{'query': 'What were the main causes for increased lead times mentioned in the report?'}, {'query': 'List the KPIs for Region B from the Q3 table.'}, {'query': 'Which parts have lead time more than 30 days?'}, {'query': 'What is the procedure for critical parts exceptions and the SLA for documenting root cause?'}]


In [30]:
results = []

for item in queries:

    q = item["query"]

    print("Processing:", q)

    output = answer_query(q)

    results.append(output)

Processing: What were the main causes for increased lead times mentioned in the report?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing: List the KPIs for Region B from the Q3 table.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing: Which parts have lead time more than 30 days?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing: What is the procedure for critical parts exceptions and the SLA for documenting root cause?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [32]:
with open(
    "answers.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("answers.json created successfully.")

answers.json created successfully.


In [33]:
for item in results:

    print("=" * 80)

    print("Question:")
    print(item["query"])

    print()

    print("Answer:")
    print(item["answer"])

    print()

    print("Retrieved Chunks:")
    print(item["retrieved_chunks"])

Question:
What were the main causes for increased lead times mentioned in the report?

Answer:
You are a question answering assistant.

Answer ONLY from the context below.

If the answer is not present, reply exactly:

I don't know from the document.

Context:
Operational Observations (cid:127) Region A experienced a 12% increase in expedited shipments due to stockouts. (cid:127) Region B showed a 7-day increase in average lead time during monsoon months. (cid:127) Spares accuracy (picks vs orders) remained at 98.2% but returns increased slightly due to labeling errors. Notes on data sources: Data aggregated from ERP, WMS, and Vendor portals. Timestamp alignment used UTC.

0 1 2 3 Metric Region A Region B Region C Avg Lead Time (days) 14 21 12 On-time Delivery (%) 93.4 88.1 95.0 Expedited Shipments (%) 6.1 12.5 4.0 Inventory Turns 3.5 2.8 4.1

Sample Parts List Part ID Description Category Qty On Hand Lead Time (days) P-1001 Avionics Control Module Control 25 30 P-1002 Navigation Senso